# Differential evolution — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sphere(x, target=None):
    x=np.asarray(x, dtype=float)
    if target is None: target=np.zeros_like(x)
    return float(np.sum((x-np.asarray(target,dtype=float))**2))
def rastrigin(x):
    x=np.asarray(x,dtype=float); return float(10*len(x)+np.sum(x*x-10*np.cos(2*np.pi*x)))
def softmax_min(vals, temp=1.0):
    vals=np.asarray(vals,dtype=float); w=np.exp(-(vals-vals.min())/temp); return w/w.sum()
print('setup ok')

## Use scaled differences between population members as self-tuned mutation steps
Differential evolution is a population optimizer for real vectors. Its signature move is simple and powerful: make a mutant by adding a scaled difference vector to a third individual, cross it with the target, and greedily keep whichever vector has lower objective value.

In [ ]:
# 1) The differential mutation vector is a scaled population difference
target=np.array([0.,0.]); a=np.array([1.,1.]); b=np.array([3.,0.]); c=np.array([0.,2.]); F=0.5
mutant=a+F*(b-c)
plt.figure(figsize=(4,4)); plt.scatter([a[0],b[0],c[0],mutant[0]],[a[1],b[1],c[1],mutant[1]],s=80); plt.annotate('mutant', mutant); plt.arrow(a[0],a[1],*(F*(b-c)),head_width=.08,length_includes_head=True,color='r'); plt.axis('equal'); plt.title('a + F(b-c)')
assert np.allclose(mutant, [2.5,0.0])

In [ ]:
# 2) Binomial crossover mixes target and mutant coordinates
target=np.array([0.,0.]); mutant=np.array([2.5,0.]); mask=np.array([True,False]); trial=np.where(mask,mutant,target)
obj=lambda z: np.sum((z-np.array([2.,1.]))**2)
plt.figure(figsize=(4,4)); plt.scatter([target[0],mutant[0],trial[0],2],[target[1],mutant[1],trial[1],1],s=80); plt.legend(['target','mutant','trial','optimum']); plt.axis('equal'); plt.title('trial inherits selected coordinates')
assert np.allclose(trial,[2.5,0.0]) and abs(obj(trial)-1.25)<1e-12

In [ ]:
# 3) Greedy replacement keeps the lower objective vector
old=np.array([0.,0.]); trial=np.array([2.5,0.]); optimum=np.array([2.,1.])
oldv=np.sum((old-optimum)**2); trialv=np.sum((trial-optimum)**2); kept=trial if trialv<oldv else old
plt.figure(figsize=(6,3)); plt.bar(['old','trial'],[oldv,trialv],color=['tab:red','tab:green']); plt.ylabel('objective'); plt.title('lower objective survives')
assert oldv==5.0 and trialv==1.25 and np.allclose(kept,trial)

In [ ]:
# 4) Differential evolution converges on a 2D sphere
rng=np.random.default_rng(8); pop=rng.uniform(-5,5,(25,2)); best=[]; target=np.array([1.,-1.])
for _ in range(35):
    for i in range(len(pop)):
        idx=rng.choice([j for j in range(len(pop)) if j!=i],3,replace=False); a,b,c=pop[idx]; mutant=a+0.7*(b-c); mask=rng.random(2)<0.8; mask[rng.integers(2)]=True; trial=np.where(mask,mutant,pop[i])
        if np.sum((trial-target)**2) < np.sum((pop[i]-target)**2): pop[i]=trial
    best.append(np.min(np.sum((pop-target)**2,axis=1)))
plt.figure(figsize=(6,3)); plt.semilogy(best); plt.xlabel('generation'); plt.ylabel('best objective'); plt.title('population differences drive convergence')
assert best[-1] < 1e-2 and best[-1] < best[0]

In [ ]:
# 5) The scale factor F changes step length
b=np.array([3.,0.]); c=np.array([0.,2.]); a=np.array([1.,1.]); Fs=np.array([0.2,0.5,1.0]); mutants=np.array([a+F*(b-c) for F in Fs]); lengths=np.linalg.norm(mutants-a,axis=1)
plt.figure(figsize=(4,4)); plt.scatter(mutants[:,0],mutants[:,1],c=Fs,s=100,cmap='viridis'); [plt.text(mutants[i,0],mutants[i,1],f'F={Fs[i]}') for i in range(len(Fs))]; plt.scatter([a[0]],[a[1]],c='k'); plt.axis('equal'); plt.title('larger F means longer differential jumps')
assert lengths[2] > lengths[1] > lengths[0]